# RET503 YOLO basket (yolo26n)
Runtime -> Change runtime type -> **T4 GPU** sebelum menjalankan. Jalankan sel berurutan.

In [ ]:
REPO_URL = "https://github.com/AkbarL090/ret503-yolo-basketball.git"   # ganti jika nama repo berbeda
!git clone $REPO_URL repo
%cd repo
!pip -q install -U ultralytics
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
!python scripts/build_yolo_dataset.py   # harus: {'train': 124, 'val': 23, 'test': 26}

## E1: pretrained (baseline)

In [ ]:
!yolo detect train data=dataset_v1/data.yaml model=yolo26n.pt epochs=30 imgsz=416 batch=16 seed=0 name=E1_pretrained

## E2: scratch (hanya bobot awal yang berbeda)

In [ ]:
!yolo detect train data=dataset_v1/data.yaml model=yolo26n.yaml pretrained=False epochs=30 imgsz=416 batch=16 seed=0 name=E2_scratch

## E3: tuning satu perubahan (imgsz 640)

In [ ]:
!yolo detect train data=dataset_v1/data.yaml model=yolo26n.pt epochs=30 imgsz=640 batch=16 seed=0 name=E3_imgsz640

## Ringkasan val (epoch terbaik) dan pilih model

In [ ]:
import pandas as pd, glob
for p in sorted(glob.glob("runs/detect/E*/results.csv")):
    d = pd.read_csv(p); d.columns = d.columns.str.strip()
    b = d.loc[d["metrics/mAP50-95(B)"].idxmax()]
    print(p.split("/")[2], "epoch", int(b["epoch"]), "mAP50 %.3f mAP50-95 %.3f" % (b["metrics/mAP50(B)"], b["metrics/mAP50-95(B)"]))

In [ ]:
!yolo detect val model=runs/detect/E1_pretrained/weights/best.pt data=dataset_v1/data.yaml imgsz=416 split=val
!yolo detect val model=runs/detect/E3_imgsz640/weights/best.pt data=dataset_v1/data.yaml imgsz=640 split=val

In [ ]:
!python scripts/benchmark.py E1_pretrained:416 E3_imgsz640:640

## Test: model terpilih saja, SEKALI

In [ ]:
!yolo detect val model=runs/detect/E3_imgsz640/weights/best.pt data=dataset_v1/data.yaml imgsz=640 split=test